# 10 — Model Comparison

**AI Financial Intelligence Platform — ML Module**

---

## Purpose

Compare all trained models across tasks to select the best candidate for production.

### Metrics Used
- **Regression**: RMSE, MAE, MAPE, R²
- **Classification**: F1, Precision, Recall, AUC-ROC
- **Anomaly Detection**: Precision@k, Recall@k
- **Ranking**: Combined score weighted by business impact

### Sections
1. Load all evaluation results
2. Comparison table — regression models
3. Comparison table — classification models
4. Comparison table — anomaly detection
5. Performance visualization
6. Production model selection
7. Model registry update


In [ ]:
import os
import json
import pickle
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

warnings.filterwarnings('ignore')

REPORTS_DIR = '../reports/'
MODELS_DIR  = '../models/'

os.makedirs(REPORTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR,  exist_ok=True)

## 1. Load All Evaluation Results


In [ ]:
# ── Load JSON evaluation reports generated by notebooks 03–09 ─────────────────
EVAL_FILES = {
    'revenue_forecasting':   'revenue_forecasting_eval.json',
    'sales_forecasting':     'sales_forecasting_eval.json',
    'inventory_prediction':  'inventory_prediction_eval.json',
    'profit_prediction':     'profit_prediction_eval.json',
    'expense_prediction':    'expense_prediction_eval.json',
    'anomaly_detection':     'anomaly_detection_eval.json',
}

eval_reports = {}
missing_reports = []

for task, filename in EVAL_FILES.items():
    path = os.path.join(REPORTS_DIR, filename)
    if os.path.exists(path):
        with open(path, 'r') as f:
            eval_reports[task] = json.load(f)
        print(f'✓ Loaded: {filename}')
    else:
        missing_reports.append(filename)
        print(f'✗ Missing: {filename}  (run notebook {task} first)')

# ── Also discover any saved model files ──────────────────────────────────────
model_files = [f for f in os.listdir(MODELS_DIR) if f.endswith('.pkl')]
print(f'\nSaved model files in {MODELS_DIR}:')
for mf in sorted(model_files):
    size_kb = os.path.getsize(os.path.join(MODELS_DIR, mf)) / 1024
    print(f'  {mf:<55} ({size_kb:>6.1f} KB)')

print(f'\nTotal reports loaded: {len(eval_reports)} / {len(EVAL_FILES)}')
if missing_reports:
    print('NOTE: Some reports are missing — run the corresponding notebooks to generate them.')
    print('Using synthetic/fallback data for demonstration.')

## 2. Regression Model Comparison


In [ ]:
# ── Collect regression metrics (RMSE, MAE, MAPE) from eval reports ────────────
REGRESSION_TASKS = [
    'revenue_forecasting',
    'sales_forecasting',
    'inventory_prediction',
    'profit_prediction',
    'expense_prediction',
]

reg_rows = []

for task in REGRESSION_TASKS:
    if task not in eval_reports:
        # Provide placeholder so the comparison table always renders
        reg_rows.append({'Task': task, 'Model': 'N/A (run notebook)',
                         'RMSE': np.nan, 'MAE': np.nan, 'MAPE_%': np.nan})
        continue

    report = eval_reports[task]

    # Different notebooks store results differently
    if 'models' in report and isinstance(report['models'], list):
        for m in report['models']:
            row = {
                'Task':   task,
                'Model':  m.get('model', m.get('Model', 'Unknown')),
                'RMSE':   m.get('RMSE',  np.nan),
                'MAE':    m.get('MAE',   np.nan),
                'MAPE_%': m.get('MAPE',  m.get('MAPE_%', np.nan)),
            }
            reg_rows.append(row)
    else:
        # Single-model report (e.g. profit_prediction)
        reg_rows.append({
            'Task':   task,
            'Model':  report.get('model', report.get('best_model', 'Unknown')),
            'RMSE':   report.get('RMSE',  np.nan),
            'MAE':    report.get('MAE',   np.nan),
            'MAPE_%': report.get('MAPE',  np.nan),
        })

reg_df = pd.DataFrame(reg_rows)
reg_df = reg_df.round({'RMSE': 2, 'MAE': 2, 'MAPE_%': 3})

print('=== Regression Model Comparison ===')
print(reg_df.to_string(index=False))

# Best model per task
best_reg = (
    reg_df.dropna(subset=['MAPE_%'])
    .sort_values('MAPE_%')
    .groupby('Task')
    .first()
    .reset_index()
)
print('\n=== Best Model Per Regression Task (by MAPE) ===')
print(best_reg[['Task', 'Model', 'RMSE', 'MAE', 'MAPE_%']].to_string(index=False))

## 3. Classification Model Comparison


In [ ]:
# ── Business Health Score uses a rule-based scoring (no classification labels yet) ─
# When labeled data becomes available, classification metrics will populate here.

# For now, display the health score distribution as a proxy for classification quality.
health_csv = os.path.join(REPORTS_DIR, 'latest_health_scores.csv')

if os.path.exists(health_csv):
    health_df = pd.read_csv(health_csv)

    print('=== Business Health Score Summary ===')
    print(health_df[['name', 'health_score', 'health_label']].to_string(index=False))

    # Label distribution
    label_dist = health_df['health_label'].value_counts()
    print('\nHealth label distribution:')
    print(label_dist.to_string())

    # When labeled data is available, extend with:
    # - GradientBoosting classifier trained on health features
    # - Precision, Recall, F1, AUC-ROC metrics
    # - Confusion matrix visualization
    print('\nNote: Classification metrics (F1/AUC) will activate when labeled training data is provided.')
else:
    print('Health score report not found — run notebook 08 first.')
    print('\nPlaceholder classification comparison table:')
    clf_placeholder = pd.DataFrame([
        {'Task': 'health_score_classification', 'Model': 'GradientBoosting',
         'Precision': 'N/A', 'Recall': 'N/A', 'F1': 'N/A', 'AUC': 'N/A'},
        {'Task': 'health_score_classification', 'Model': 'XGBoost',
         'Precision': 'N/A', 'Recall': 'N/A', 'F1': 'N/A', 'AUC': 'N/A'},
    ])
    print(clf_placeholder.to_string(index=False))

## 4. Anomaly Detection Comparison


In [ ]:
# ── Load anomaly detection evaluation results ─────────────────────────────────
if 'anomaly_detection' in eval_reports:
    ad_report = eval_reports['anomaly_detection']
    ad_df = pd.DataFrame(ad_report.get('method_comparison', []))

    print('=== Anomaly Detection Method Comparison ===')
    print(ad_df.to_string(index=False))
    print(f"\nHigh-confidence anomalies (≥3 methods): {ad_report.get('high_confidence_anomalies', 'N/A')}")
    print(f"Contamination rate: {ad_report.get('contamination_rate', 'N/A')}")
    print(f"Transactions analyzed: {ad_report.get('n_transactions', 'N/A'):,}")
else:
    print('Anomaly detection report not found — run notebook 09 first.')
    ad_df = pd.DataFrame([
        {'Method': 'zscore_flag',     'Flagged': 'N/A', 'Flag_Rate_%': 'N/A'},
        {'Method': 'iqr_flag',        'Flagged': 'N/A', 'Flag_Rate_%': 'N/A'},
        {'Method': 'if_flag',         'Flagged': 'N/A', 'Flag_Rate_%': 'N/A'},
        {'Method': 'lof_flag',        'Flagged': 'N/A', 'Flag_Rate_%': 'N/A'},
        {'Method': 'ae_flag',         'Flagged': 'N/A', 'Flag_Rate_%': 'N/A'},
    ])
    print(ad_df.to_string(index=False))

# Load detected anomalies CSV if available
anomaly_csv = os.path.join(REPORTS_DIR, 'detected_anomalies.csv')
if os.path.exists(anomaly_csv):
    anomalies = pd.read_csv(anomaly_csv)
    print(f'\nTop high-confidence anomalies ({len(anomalies)} total):')
    print(anomalies.head(10).to_string(index=False))

## 5. Performance Visualization


In [ ]:
# ── Bar chart: MAPE comparison across regression tasks ─────────────────────────
reg_plot = reg_df.dropna(subset=['MAPE_%']).copy()

if not reg_plot.empty:
    # Pivot: task × model
    reg_pivot = reg_plot.pivot_table(index='Task', columns='Model', values='MAPE_%', aggfunc='min')

    fig, ax = plt.subplots(figsize=(13, 5))
    reg_pivot.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='white')
    ax.set_title('MAPE (%) by Task and Model — Lower is Better', fontweight='bold')
    ax.set_ylabel('MAPE (%)')
    ax.set_xlabel('Forecasting Task')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(title='Model', bbox_to_anchor=(1.01, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

    # ── Grouped bar: RMSE side-by-side ────────────────────────────────────────────
    fig2, ax2 = plt.subplots(figsize=(13, 5))
    rmse_pivot = reg_plot.pivot_table(index='Task', columns='Model', values='RMSE', aggfunc='min')
    rmse_pivot.plot(kind='bar', ax=ax2, colormap='tab10', edgecolor='white')
    ax2.set_title('RMSE by Task and Model', fontweight='bold')
    ax2.set_ylabel('RMSE')
    ax2.set_xlabel('Forecasting Task')
    ax2.tick_params(axis='x', rotation=30)
    ax2.legend(title='Model', bbox_to_anchor=(1.01, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
else:
    print('No regression evaluation data available for visualization.')
    print('Run notebooks 03–07 first to generate evaluation reports.')

# ── Radar chart: multi-metric model performance ────────────────────────────────
# Build a normalized performance radar for each task's best model
if not reg_plot.empty:
    # Normalize metrics to 0-100 (0 = worst, 100 = best)
    best_per_task = (
        reg_plot.sort_values('MAPE_%')
        .groupby('Task')
        .first()
        .reset_index()
    )

    metrics = ['RMSE', 'MAE', 'MAPE_%']
    for m in metrics:
        mn, mx = best_per_task[m].min(), best_per_task[m].max()
        if mx > mn:
            # Invert: lower metric = better = higher score
            best_per_task[f'{m}_score'] = 100 * (1 - (best_per_task[m] - mn) / (mx - mn))
        else:
            best_per_task[f'{m}_score'] = 100

    score_cols_radar = [f'{m}_score' for m in metrics]
    radar_data = best_per_task[['Task'] + score_cols_radar].set_index('Task')

    fig3, ax3 = plt.subplots(figsize=(10, 5))
    radar_data.T.plot(kind='bar', ax=ax3, colormap='Set3', edgecolor='white')
    ax3.set_title('Normalized Performance Score (100 = best) — Best Model per Task',
                   fontweight='bold')
    ax3.set_ylabel('Score (0–100)')
    ax3.set_ylim(0, 110)
    ax3.legend(title='Task', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()

## 6. Production Model Selection


In [ ]:
# ── Business-impact-weighted model scoring ────────────────────────────────────
# Weight each task by business impact on the platform
TASK_WEIGHTS = {
    'revenue_forecasting':  0.30,  # Highest impact — drives P&L planning
    'profit_prediction':    0.25,  # Directly tied to bottom line
    'expense_prediction':   0.20,  # Cost control
    'sales_forecasting':    0.15,  # Operational / inventory link
    'inventory_prediction': 0.10,  # Operational efficiency
}

print('=== Production Model Selection ===')
print(f'{'Task':<30} {'Best Model':<25} {'MAPE_%':>8}  {'Impact Weight':>14}')
print('-' * 80)

selected_models = {}
weighted_score_total = 0.0

for task, weight in TASK_WEIGHTS.items():
    task_rows = reg_df[(reg_df['Task'] == task) & reg_df['MAPE_%'].notna()]
    if task_rows.empty:
        model_name = 'N/A'
        mape_val   = np.nan
        score_contribution = 0.0
    else:
        best_row   = task_rows.loc[task_rows['MAPE_%'].idxmin()]
        model_name = best_row['Model']
        mape_val   = best_row['MAPE_%']
        # Score contribution: 100 − MAPE (capped at 0) × weight
        score_contribution = max(0, 100 - mape_val) * weight
        weighted_score_total += score_contribution

    selected_models[task] = {
        'model': model_name,
        'mape_pct': float(mape_val) if not np.isnan(mape_val) else None,
        'weight': weight,
    }
    mape_str = f'{mape_val:.2f}%' if not np.isnan(mape_val) else 'N/A'
    print(f'{task:<30} {model_name:<25} {mape_str:>8}  {weight:>14.0%}')

print('-' * 80)
print(f'Platform Weighted Forecast Quality Score: {weighted_score_total:.1f} / 100')

# Anomaly detection — select by lowest flag rate (closest to expected contamination)
anomaly_best = 'IsolationForest'
if 'anomaly_detection' in eval_reports:
    ad_methods = eval_reports['anomaly_detection'].get('method_comparison', [])
    if ad_methods:
        # Prefer the method whose flag rate is closest to contamination rate
        contamination_pct = eval_reports['anomaly_detection'].get('contamination_rate', 0.01) * 100
        closest = min(ad_methods,
                      key=lambda m: abs(m.get('Flag_Rate_%', 99) - contamination_pct)
                      if isinstance(m.get('Flag_Rate_%'), (int, float)) else 99)
        anomaly_best = closest.get('Method', 'IsolationForest')

selected_models['anomaly_detection'] = {
    'model': anomaly_best,
    'ensemble': 'voting (IF + LOF + statistical baseline)',
}
selected_models['business_health_score'] = {
    'model': 'WeightedKPIScoring',
    'version': '1.0',
}

print('\n=== Final Selected Models ===')
for task, info in selected_models.items():
    print(f'  {task:<30}: {info["model"]}')

## 7. Model Registry Update


In [ ]:
# ── Map task → artifact file path ─────────────────────────────────────────────
MODEL_ARTIFACT_MAP = {
    'revenue_forecasting':   'revenue_forecasting_xgboost.pkl',
    'sales_forecasting':     'sales_forecasting_xgb_product.pkl',
    'inventory_prediction':  'inventory_demand_xgboost.pkl',
    'profit_prediction':     'profit_prediction_xgboost.pkl',
    'expense_prediction':    'expense_prediction_xgboost.pkl',
    'anomaly_detection':     'anomaly_detection_isolation_forest.pkl',
    'business_health_score': 'health_score_pipeline.pkl',
}

# Build registry entry for each task
registry = {}
import datetime as dt

for task, info in selected_models.items():
    artifact = MODEL_ARTIFACT_MAP.get(task, 'unknown.pkl')
    artifact_path = os.path.join(MODELS_DIR, artifact)
    exists = os.path.exists(artifact_path)

    registry[task] = {
        'model_name':        info['model'],
        'artifact_file':     artifact,
        'artifact_path':     os.path.abspath(artifact_path),
        'artifact_exists':   exists,
        'registered_at':     dt.datetime.utcnow().isoformat(),
        'status':            'production' if exists else 'pending_training',
        'metrics':           {k: v for k, v in info.items() if k not in ['model']},
    }
    status_icon = '✓' if exists else '✗'
    print(f'{status_icon} {task:<30}: {info["model"]} → {artifact}')

# ── Write model_registry.json ─────────────────────────────────────────────────
registry_path = os.path.join(MODELS_DIR, 'model_registry.json')
with open(registry_path, 'w') as f:
    json.dump(registry, f, indent=2, default=str)
print(f'\n✓ Model registry saved → {registry_path}')

# ── Write a human-readable Markdown summary ───────────────────────────────────
md_lines = [
    '# AI Financial Intelligence Platform — Model Registry',
    '',
    f'**Generated:** {dt.datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC")}',
    f'**Platform Forecast Quality Score:** {weighted_score_total:.1f} / 100',
    '',
    '| Task | Model | Status | MAPE (%) |',
    '|------|-------|--------|----------|',
]
for task, entry in registry.items():
    mape_str = f"{entry['metrics'].get('mape_pct', 'N/A')}"
    md_lines.append(
        f"| {task} | {entry['model_name']} | "
        f"{entry['status']} | {mape_str} |"
    )

md_path = os.path.join(REPORTS_DIR, 'model_registry.md')
with open(md_path, 'w') as f:
    f.write('\n'.join(md_lines))
print(f'✓ Model registry Markdown → {md_path}')

print('\nModel comparison and registry update complete.')